使用先前检查点的配置调用图以从该点重放。

In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, NotRequired
from langchain_core.utils.uuid import uuid7

class State(TypedDict):
    topic: NotRequired[str]
    joke: NotRequired[str]


def generate_topic(state: State):
    return {"topic": "socks in the dryer"}


def write_joke(state: State):
    return {"joke": f"Why do {state['topic']} disappear? They elope!"} # type: ignore


checkpointer = InMemorySaver()
graph = (
    StateGraph(State)
    .add_node("generate_topic", generate_topic)
    .add_node("write_joke", write_joke)
    .add_edge(START, "generate_topic")
    .add_edge("generate_topic", "write_joke")
    .add_edge("write_joke", END)
    .compile(checkpointer=checkpointer)
)

In [2]:
# Step 1: Run the graph
config = {"configurable": {"thread_id": str(uuid7())}}
result = graph.invoke({}, config) # type: ignore
result

{'topic': 'socks in the dryer',
 'joke': 'Why do socks in the dryer disappear? They elope!'}

In [3]:
# Step 2: Find a checkpoint to replay from
history = list(graph.get_state_history(config)) # type: ignore
# History is in reverse chronological order
for state in history:
    print(f"next={state.next}, checkpoint_id={state.config['configurable']['checkpoint_id']}") # type: ignore

next=(), checkpoint_id=1f146285-6994-685c-8002-382bbcdf6a29
next=('write_joke',), checkpoint_id=1f146285-6993-6b0a-8001-f55604856831
next=('generate_topic',), checkpoint_id=1f146285-6992-6070-8000-d988b05f5b56
next=('__start__',), checkpoint_id=1f146285-6990-68d8-bfff-14b4c840fe17


In [7]:
# Step 3: Replay from a specific checkpoint
# Find the checkpoint before write_joke
before_joke = next(s for s in history if s.next == ("write_joke",))
before_joke.values, before_joke.config

({'topic': 'socks in the dryer'},
 {'configurable': {'thread_id': '019de8d0-27eb-7230-b230-ee3dd64dc549',
   'checkpoint_ns': '',
   'checkpoint_id': '1f146285-6993-6b0a-8001-f55604856831'}})

In [8]:
replay_result = graph.invoke(None, before_joke.config)
replay_result

{'topic': 'socks in the dryer',
 'joke': 'Why do socks in the dryer disappear? They elope!'}